# EDA: Online Retail Transactions

Exploratory Data Analysis on the Online Retail dataset to identify data quality issues and define cleaning rules for the ETL pipeline.

**Dataset:** online_retail.csv (Kaggle)  
**Rows:** 541,909  
**Columns:** InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("EDA").getOrCreate()

df_raw = spark.read.csv("/home/jovyan/work/data/raw/online_retail.csv", header=True, inferSchema=True)
df_raw.printSchema()
print(f"Total rows: {df_raw.count()}")

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)

Total rows: 541909


In [6]:
df_raw.show(10, truncate=False)

+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |2010-12-01 08:26:00|2.55     |17850.0   |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |2010-12-01 08:26:00|2.75     |17850.0   |United Kingdom|
|536365   |84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
|536365   |84029E   |RED WOOLLY HOTTIE WHITE HEART.     |6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|
|536365   |22752

In [8]:
from pyspark.sql import functions as F

In [10]:
null_counts = df_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) 
    for c in df_raw.columns
])
null_counts.show()

+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|        0|        0|       1454|       0|          0|        0|    135080|      0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+



In [12]:
print(f"Quantity <= 0: {df_raw.filter(F.col('Quantity') <= 0).count()}")
print(f"UnitPrice <= 0: {df_raw.filter(F.col('UnitPrice') <= 0).count()}")

Quantity <= 0: 10624
UnitPrice <= 0: 2517


In [13]:
cancellations = df_raw.filter(F.col("InvoiceNo").startswith("C"))
print(f"Cancellations: {cancellations.count()}")

Cancellations: 9288


In [14]:
total = df_raw.count()
distinct = df_raw.dropDuplicates(["InvoiceNo", "StockCode", "InvoiceDate"]).count()
print(f"Duplicate rows: {total - distinct}")

Duplicate rows: 10677


In [16]:
df_raw.describe(["Quantity", "UnitPrice"]).show()

+-------+------------------+-----------------+
|summary|          Quantity|        UnitPrice|
+-------+------------------+-----------------+
|  count|            541909|           541909|
|   mean|  9.55224954743324|4.611113626090092|
| stddev|218.08115785023446|96.75985306117967|
|    min|            -80995|        -11062.06|
|    max|             80995|          38970.0|
+-------+------------------+-----------------+



In [19]:
df_raw.groupBy("Country").count().orderBy("count", ascending=False).show(10)

+--------------+------+
|       Country| count|
+--------------+------+
|United Kingdom|495478|
|       Germany|  9495|
|        France|  8557|
|          EIRE|  8196|
|         Spain|  2533|
|   Netherlands|  2371|
|       Belgium|  2069|
|   Switzerland|  2002|
|      Portugal|  1519|
|     Australia|  1259|
+--------------+------+
only showing top 10 rows



## DQ Summary

| Issue | Count | % of Total | Action |
|---|---|---|---|
| NULL Description | 1,454 | 0.27% | Keep (not critical) |
| NULL CustomerID | 135,080 | 24.9% | Keep (transaction is valid) |
| Quantity <= 0 | 10,624 | 1.96% | Remove |
| UnitPrice <= 0 | 2,517 | 0.46% | Remove |
| Cancellations (C prefix) | 9,288 | 1.71% | Remove |
| Duplicates | 10,677 | 1.97% | Remove |

## Cleaning Rules for Silver Layer

1. Drop rows with NULL in: InvoiceNo, StockCode, Quantity, UnitPrice, InvoiceDate
2. Drop rows where Quantity <= 0
3. Drop rows where UnitPrice <= 0
4. Drop cancellations (InvoiceNo starts with "C")
5. Drop duplicates by InvoiceNo + StockCode + InvoiceDate
6. Cast types: InvoiceDate → timestamp, Quantity → int, UnitPrice → double
7. Add computed column: Amount = Quantity × UnitPrice
8. Clean text: trim(Description), uppercase(StockCode)

## Observations

- CustomerID is 25% NULL — high but acceptable since we don't filter on it
- Cancellations and negative Quantity overlap heavily (~9,288 of 10,624)
- Both Quantity and UnitPrice have extreme outliers (stddev >> mean), largely driven by negatives and bulk orders
- After cleaning, expect ~520k+ clean rows